In [7]:
import os
from dotenv import load_dotenv
import gradio as gr 
from openai import OpenAI

In [20]:
load_dotenv(override=True)
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY is not set in the environment variables.")
if not api_key.startswith("gsk_"):
    raise ValueError("GEMINI_API_KEY does not start with 'gsk_'. Please check your API key.")
else:
    print("Groq api key is set correctly.")

Groq api key is set correctly.


In [25]:
gemini = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")
groq_model = "openai/gpt-oss-20b"

In [22]:
system_message = """You are helpful assistant that translates natural language into SQL queries. You will be given a natural language question and you should generate a SQL query that answers the question. The SQL query should be valid and executable on a database. Do not include any explanations or additional text, only provide the SQL query."""

In [26]:
def chat_bot(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = gemini.chat.completions.create(model = groq_model, messages = messages, stream=True)
    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response
            

In [27]:
gr.ChatInterface(
    fn=chat_bot,
    title="SQL Query Generator",
    type="messages",
    description="Enter a natural language question and get the corresponding SQL query.",
    flagging_mode="never"
).launch(share=True)

* Running on local URL:  http://127.0.0.1:7872

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [28]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [30]:
gr.ChatInterface(fn=chat_bot, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
